In [ ]:
import os
import fastf1

os.makedirs("cache", exist_ok=True)
fastf1.Cache.enable_cache("cache")

In [ ]:
session = fastf1.get_session(2025, "Monza", "Q")
session.load()
print("loaded")

In [ ]:
lap = session.laps.pick_fastest()
print(lap["Driver"], lap["LapTime"])

tel = lap.get_telemetry()
print(tel.columns.tolist())
print(tel.shape)
print(tel.head(3))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. lap length from the Distance column
print("last distance (m):", tel["Distance"].iloc[-1])

# 2. lap length from the X, Y positions (in their own units)
dx = np.diff(tel["X"])
dy = np.diff(tel["Y"])
path_units = np.sqrt(dx**2 + dy**2).sum()
print("path length from X,Y (units):", path_units)
print("units per metre:", path_units / tel["Distance"].iloc[-1])

# 3. two plots
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(tel["Distance"], tel["Speed"])
ax[0].set_xlabel("Distance (m)")
ax[0].set_ylabel("Speed (km/h)")
ax[1].plot(tel["X"], tel["Y"])
ax[1].set_aspect("equal")
ax[1].set_title("Track shape")
plt.show()

In [ ]:
out = tel[["Distance", "Speed", "Throttle", "Brake", "nGear", "RPM"]].copy()
out["t_s"]  = tel["Time"].dt.total_seconds()   # time in seconds from lap start
out["x_m"]  = tel["X"] / 10                    # X in metres
out["y_m"]  = tel["Y"] / 10                    # Y in metres
out["v_ms"] = tel["Speed"] / 3.6               # speed in m/s
out["Brake"] = tel["Brake"].astype(int)        # True/False to 1/0

print(out.isna().sum())     # missing values per column, expect all 0
print(out.head(3))

os.makedirs("data", exist_ok=True)
out.to_csv("data/monza_2025Q_VER_fastest.csv", index=False)
print("saved", out.shape)